In [1]:
import os
os.chdir(r"C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2")
print(os.getcwd())

C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2


## Step 1: Load Dataset

In [2]:
import pandas as pd
df = pd.read_csv("data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


## Step 2: Add Ground Truth Columns

In [3]:
# 1. Define rule sets
RULES = [
    {
        "keywords": ["security", "alert", "invoice", "overdue", "urgent"],
        "intent": "notify",
        "tone": "urgent"
    },
    {
        "keywords": ["meeting", "schedule", "report", "project"],
        "intent": "respond",
        "tone": "neutral"
    },
    {
        "keywords": ["promotion", "sale", "newsletter", "congratulations"],
        "intent": "ignore",
        "tone": "neutral"
    },
    {
        "keywords": ["thanks", "please"],
        "intent": "ignore",
        "tone": "polite"
    }
]
# 2. Labeling function using rule iteration
def auto_label_v2(text):
    text = str(text).lower()
    for rule in RULES:
        if any(word in text for word in rule["keywords"]):
            return rule["intent"], rule["tone"]
    # Default fallback
    return "respond", "neutral"
# 3. Apply labels
df["ideal_intent"], df["ideal_tone"] = zip(*df["body"].apply(auto_label_v2))
# 4. Save updated dataset
df.to_csv("data/sample_emails_with_triage_200.csv", index=False)
print("Success! File updated with 200 labels.")
print(df[["body", "ideal_intent", "ideal_tone"]].head())

Success! File updated with 200 labels.
                                                body ideal_intent ideal_tone
0  Reminder: The client meeting is scheduled at 1...      respond    neutral
1  Your invoice of INR 25515.09 is due on 2025-12...       notify     urgent
2  Reminder: The client meeting is scheduled at 1...      respond    neutral
3  Hello team, please find the attached weekly re...      respond    neutral
4  Hello team, please find the attached weekly re...      respond    neutral


## Step 3: Email Assistant Logic (Reuse from Milestone 1)

In [4]:
def email_assistant(body):
    text = body.lower()
    if "invoice" in text:
        return "respond", "urgent"
    if "meeting" in text:
        return "respond", "polite"
    if "internship" in text:
        return "notify", "polite"
    if "query" in text:
        return "respond", "neutral"
    if "newsletter" in text:
        return "ignore", "neutral"
    return "ignore", "neutral"

## Step 4: Generate Predictions

In [5]:
predictions = []
for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })
pred_df = pd.DataFrame(predictions)
pred_df.head()

,id,predicted_intent,predicted_tone
0,1,respond,polite
1,2,respond,urgent
2,3,respond,polite
3,4,ignore,neutral
4,5,ignore,neutral


## Step 5: Evaluate Accuracy

In [6]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

In [7]:
eval_df = df.merge(pred_df, on="id")
eval_df["score"] = eval_df.apply(evaluate, axis=1)
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
print(f"Overall Accuracy: {accuracy:.2f}%")

Overall Accuracy: 55.00%


## Step 6: Save Evaluation Output

In [8]:
eval_df.to_csv(
    "data/milestone2_output_pranaya.csv",
    index=False
)